# 28_02 고장 예측 결과 해석

In [1]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import os
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np


try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 851 ('font.family : Malgun Gothic')
Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 852 ('axes.unicode_minus : False')


✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


In [2]:
import sys
print(sys.executable)

c:\Users\mzlap\Desktop\KNA-Data-analysis-1st\.venv\Scripts\python.exe


In [3]:
# 28_1장 기본 머닝러신 코드 재현
df = pd.read_csv("28_cmapss_fd001_sample.csv")
feature_cols = ["sensor_2",	"sensor_3",	"sensor_4",	"sensor_7",	"sensor_11",	"sensor_15"]

X = df[feature_cols]
y = df["failure_soon"]

# X.head()


# 분할은 2:8로 하는 것이 좋음



In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42).fit(X_train , y_train)

# model 

y_pred = model.predict(X_test)

y_pred[:15]

array([0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

# 01 F1-score와 분류 리포트
F1-score란 무엇인가 · 분류 리포트 읽는 순서


### F1 출력
고장 임박 클래스의 F1을 함수 한 줄로 계산해 출력


In [5]:
# 코드

from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("고장임박 F1 :", round(f1, 4))



고장임박 F1 : 0.675


### 리포트 출력과 해석
클래스별 지표를 표로 출력하고 한 줄씩 해석 적기


In [6]:
# 코드

from sklearn.metrics import classification_report

# report = classification_report(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names = ["정상", "고장임박"])

print(report)

              precision    recall  f1-score   support

          정상       0.91      0.93      0.92       158
        고장임박       0.71      0.64      0.68        42

    accuracy                           0.87       200
   macro avg       0.81      0.79      0.80       200
weighted avg       0.87      0.87      0.87       200



# 02 임계값 조정
임계값을 조정한다는 것 · 임계값과 정밀도·재현율


### 확률 추출과 임계값 적용
고장(1) 확률을 꺼내 임계값별 예측 생성


In [7]:
# 코드

proba = model.predict_proba(X_test)[:, 1]

# proba 출력
# array([0.03, 0.  , 0.03, 0.76, 0.29, 0.37, 0.02, 0.01, 0.  , 0.85, 0.43,
#        0.05, 0.02, 0.27, 0.03, 0.05, 0.02, 0.53, 0.02, 0.89, 0.57, 0.12,
#        0.  , 0.29, 0.06, 0.69, 0.  , 0.59, 0.44, 0.46, 0.05, 0.03, 0.1 ,
#        0.59, 0.07, 0.  , 0.17, 0.23, 0.  , 0.64, 0.05, 0.27, 0.79, 0.  ,
#        0.42, 0.02, 0.29, 0.  , 0.01, 0.09, 0.09, 0.36, 0.7 , 0.21, 0.04,
#        0.7 , 0.61, 0.03, 0.  , 0.01, 0.53, 0.33, 0.03, 0.  , 0.02, 0.88,
#        0.09, 0.05, 0.78, 0.  , 0.03, 0.  , 0.  , 0.  , 0.02, 0.12, 0.07,
#        0.  , 0.04, 0.29, 0.  , 0.01, 0.11, 0.06, 0.08, 0.22, 0.35, 0.45,
#        0.03, 0.26, 0.  , 0.  , 0.01, 0.  , 0.51, 0.02, 0.82, 0.19, 0.11,
#        0.2 , 0.01, 0.  , 0.38, 0.5 , 0.86, 0.33, 0.1 , 0.15, 0.14, 0.09,
#        0.19, 0.13, 0.07, 0.01, 0.05, 0.  , 0.1 , 0.11, 0.43, 0.36, 0.91,
#        0.06, 0.09, 0.55, 0.1 , 0.39, 0.5 , 0.02, 0.01, 0.53, 0.56, 0.01,
#        0.  , 0.34, 0.02, 0.  , 0.01, 0.29, 0.  , 0.  , 0.46, 0.79, 0.72,
#        0.  , 0.1 , 0.  , 0.  , 0.02, 0.01, 0.03, 0.43, 0.01, 0.25, 0.6 ,
#        0.17, 0.03, 0.22, 0.  , 0.03, 0.52, 0.52, 0.14, 0.09, 0.5 , 0.  ,
#        0.03, 0.  , 0.02, 0.75, 0.36, 0.09, 0.01, 0.14, 0.11, 0.19, 0.21,
#        0.54, 0.  , 0.  , 0.02, 0.52, 0.08, 0.2 , 0.01, 0.1 , 0.1 , 0.8 ,
#        0.78, 0.02, 0.01, 0.81, 0.  , 0.12, 0.66, 0.  , 0.7 , 0.87, 0.11,
#        0.12, 0.7 ])

# 0.5 이상인 값들이 1로 나올 확률이 높음


for t in [0.3, 0.5, 0.7]:
    pred_t = (proba >= t).astype(int) # 확률이 t보다 큰 값들이 몇개인지 따져보는 코드
    print(f"전체 데이터 중{t} 이상인 경우는 {pred_t.sum()}개")
    # 전체 데이터 중0.3 이상인 경우는 59개
    # 전체 데이터 중0.5 이상인 경우는 41개
    # 전체 데이터 중0.7 이상인 경우는 20개







전체 데이터 중0.3 이상인 경우는 59개
전체 데이터 중0.5 이상인 경우는 41개
전체 데이터 중0.7 이상인 경우는 20개


### 임계값별 지표 비교
각 임계값의 재현율·정밀도·F1 출력


In [8]:
# 코드
from sklearn.metrics import precision_score, recall_score, f1_score

for t in [0.3, 00.5, 0.5]:
    pt = (proba >= t).astype(int)

    print(t, round(recall_score(y_test, pt), 3),
          round(precision_score(y_test, pt), 3))
    # 가운데 값이 재현율
    # 0.3 0.81 0.576
    # 0.5 0.69 0.707
    # 0.5 0.69 0.707

0.3 0.81 0.576
0.5 0.69 0.707
0.5 0.69 0.707


### 비교표 생성
임계값별 지표와 FP·FN을 한 표로


In [9]:
# 코드
from sklearn.metrics import confusion_matrix, recall_score, precision_score
rows = []

for t in [0.2, 0.3, 0.4, 0.5, 0.7]:
    pt = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pt).ravel()
    rows.append([t, recall_score(y_test, pt), fp, fn])

print(pd.DataFrame(rows, columns = ["임계값", "재현율", "FP", "FN"]))

#   임계값  재현율   FP  FN
# 0  0.2  0.928571  36   3
# 1  0.3  0.809524  25   8
# 2  0.4  0.738095  18  11
# 3  0.5  0.690476  12  13
# 4  0.7  0.452381   1  23

# 임계값을 낮추는 방향, 재현율을 높이고, FN을 낮추는 방향
# FP를 최소

   임계값       재현율  FP  FN
0  0.2  0.928571  36   3
1  0.3  0.809524  25   8
2  0.4  0.738095  18  11
3  0.5  0.690476  12  13
4  0.7  0.452381   1  23


# 03 이상탐지 평가와 결과 해석
이상탐지 결과 평가 · 정비 의사결정 해석


### IsolationForest 실행
MIMII 특징으로 이상탐지 학습·예측


In [10]:
# 코드

from sklearn.ensemble import IsolationForest

dfm = pd.read_csv("28_mimii_features_sample.csv")
# dfm.head()

Xm = dfm[["rms", "spectral_centroid", "zero_crossing_rate"]]
# Xm.head()

# raw = IsolationForest(contamination = 0.1, random_state = 42)
raw = IsolationForest(contamination = 0.1, random_state = 42).fit_predict(Xm)
# raw

raw[:25]



array([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1])

### 변환과 평가
-1을 1(이상)로 바꿔 라벨과 비교


In [11]:
# 코드

# y_true = dfm["label"]
# # y_true.head()

# y_iso = (raw == -1).astype(int)  # -1(이상)  --> 1 / 1(정상)  --> 0 으로 변환
# y_iso[:25]
# # array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
# #        0, 0, 0]) 
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
dfm = pd.read_csv("28_mimii_features_sample.csv")
raw = IsolationForest(contamination = 0.1, random_state = 42).fit_predict(Xm)
y_true = dfm["label"]
y_iso = (raw == -1).astype(int) # -1(이상)->1, 1(정상)->0
print(confusion_matrix(y_true, y_iso))
print(classification_report(y_true, y_iso, target_names=["정상","이상"]))


[[591  36]
 [129  44]]
              precision    recall  f1-score   support

          정상       0.82      0.94      0.88       627
          이상       0.55      0.25      0.35       173

    accuracy                           0.79       800
   macro avg       0.69      0.60      0.61       800
weighted avg       0.76      0.79      0.76       800



### MIMII 로드와 분할
MIMII 특징과 라벨로 학습 준비 — 지도학습 모드


In [19]:
# 코드

from sklearn.model_selection import train_test_split

dfm = pd.read_csv("28_mimii_features_sample.csv")
# dfm.head()

Xm = dfm[["rms",	"spectral_centroid",	"zero_crossing_rate"]]
ym = dfm[["label"]]

X_train, X_test, y_train, y_test = train_test_split(Xm, ym, test_size = 0.3, random_state = 42, stratify = ym)

y_train.head()



,label
103,0
29,0
468,0
443,0
309,0


### 학습과 평가
RandomForest 학습 후 리포트 출력 — 동일 절차


In [25]:
# 코드

# from sklearn.ensemble import RandomForestClassifier

# clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

# # clf

# pred = clf.predict(X_test)
# pred[:15]  # 올바른 결과값인지 실제 제대로된 결과값과 비교해봐야함

#-------------------------------------------------------------------------------

# 레포트를 만들어야함
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

# clf

m_pred = clf.predict(X_test)
# pred[:15]
print(classification_report(y_test, m_pred, target_names=["정상", "이상"]))

# 확인 질문 · 정확도가 아니라 이상(1) 줄의 재현율 중심으로 해석


              precision    recall  f1-score   support

          정상       0.97      0.99      0.98       188
          이상       0.98      0.88      0.93        52

    accuracy                           0.97       240
   macro avg       0.97      0.94      0.96       240
weighted avg       0.97      0.97      0.97       240



c:\Users\mzlap\Desktop\KNA-Data-analysis-1st\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


### 평가 함수 정의
모델과 평가 데이터를 받는 `evaluate()` 함수


In [26]:
# 코드
from sklearn.metrics import recall_score, precision_score, confusion_matrix

def evaluate(model, X_test, y_test, name="모델"):
    pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(name, "재현율", round(recall_score(y_test, pred), 3), "FN", fn)
    


### 함수로 한 번에 평가
함수에 모델을 넣어 평가 실행 — 모델만 바꿔도 작동


In [27]:
# 코드
from sklearn.metrics import recall_score, precision_score, confusion_matrix
          
evaluate(clf, X_test, y_test, "RandomForest")

from sklearn.metrics import classification_report

print(classification_report(y_test, m_pred, target_names=["정상", "고장임박"]))


RandomForest 재현율 0.885 FN 6
              precision    recall  f1-score   support

          정상       0.97      0.99      0.98       188
        고장임박       0.98      0.88      0.93        52

    accuracy                           0.97       240
   macro avg       0.97      0.94      0.96       240
weighted avg       0.97      0.97      0.97       240



### 두 임계값 예측 생성
임계값 0.5와 0.3으로 예측을 만들고 재현율 비교


In [29]:
# 코드
proba = clf.predict_proba(X_test)[:, 1]
# proba[:25

pred_05 = (proba >= 0.5).astype(int)
pred_03 = (proba >= 0.3).astype(int)

# print(pred_05.sum())
# print(pred_03.sum())

from sklearn.metrics import recall_score
print("0.5", round(recall_score(y_test, pred_05), 3))
print("0.3", round(recall_score(y_test, pred_03), 3))

0.5 0.885
0.3 0.962


### 리포트 초안 프롬프트
평가 수치를 정비 리포트 초안으로 정리하는 AI 프롬프트


 prompt = f"""
너는 설비 정비팀에게 보고할 예지보전 리포트를 작성하는 데이터 분석가야.
아래 평가 결과를 바탕으로 정비팀이 바로 읽고 판단할 수 있는 리포트 초안을 작성해줘.

[1. CMAPSS 고장 예측 모델 - 분류 리포트 (임계값 0.5)]
{report}

[2. 임계값별 재현율/정밀도/오탐/미탐 비교]
{pd.DataFrame(rows, columns=["임계값", "재현율", "FP", "FN"]).to_markdown()}

[3. MIMII 이상탐지 비교]
- 비지도(IsolationForest): 이상 재현율 0.25, 정밀도 0.55
- 지도학습(RandomForest): 이상 재현율 0.94, 정밀도 0.97

요구사항:
1. 현재 임계값(0.5) 기준 모델 성능을 3줄로 요약
2. 임계값을 낮췄을 때(예: 0.3) 얻는 것과 잃는 것을 정비 비용 관점에서 설명
   (FN=미탐=고장을 놓쳐서 생기는 다운타임 비용, FP=오탐=불필요한 점검 비용)
3. 비지도(IsolationForest)와 지도학습(RandomForest) 중 실제 현장에 어떤 걸 권장할지와 그 이유
4. 정비팀에게 권장하는 최종 임계값과 한 줄 근거
"""
print(prompt)

# 설비 예지보전 모델 평가 및 정비 운영 권고 보고서

## 1. 분석 목적

본 분석은 설비 센서 데이터를 이용한 고장 예측 및 이상탐지 모델의 성능을 평가하고, 실제 정비 현장에서 사용할 수 있는 판단 기준을 제시하는 것을 목적으로 한다.

CMAPSS 데이터에서는 RandomForest 기반 고장 임박 예측 모델을 평가하고, 예측 임계값 변화에 따른 재현율과 오탐(FP), 미탐(FN)의 변화를 비교하였다. 또한 MIMII 데이터에서는 비지도학습 방식인 IsolationForest와 지도학습 방식인 RandomForest의 이상탐지 성능을 비교하였다. CMAPSS 모델은 `sensor_2`, `sensor_3`, `sensor_4`, `sensor_7`, `sensor_11`, `sensor_15`를 입력 변수로 사용하고 `failure_soon`을 예측 대상으로 사용하였다.

---

## 2. CMAPSS 고장 예측 모델 평가

### 2.1 기본 모델 성능

기본 RandomForest 모델의 테스트 데이터는 총 200건으로 정상 158건, 고장 임박 42건으로 구성되어 있다.

고장 임박 클래스의 성능은 다음과 같다.

| 평가 항목           |   결과 |
| --------------- | ---: |
| 전체 Accuracy     | 0.87 |
| 고장 임박 Precision | 0.71 |
| 고장 임박 Recall    | 0.64 |
| 고장 임박 F1-score  | 0.68 |
| 정상 Recall       | 0.93 |

실제 분류 리포트에서 고장 임박 클래스는 정밀도 0.71, 재현율 0.64, F1-score 0.68을 기록했으며 전체 정확도는 0.87이었다.

### 현재 성능 3줄 요약

1. 전체 정확도는 **87%**로 정상과 고장 임박 상태를 전반적으로 비교적 잘 구분하고 있다.
2. 그러나 고장 임박 재현율이 **64%** 수준이므로 실제 고장 임박 사례 중 약 36%를 놓칠 가능성이 있다.
3. 따라서 예지보전 목적에서는 전체 정확도보다 **고장 미탐(FN)을 줄이기 위한 재현율 개선**이 더 중요한 과제로 판단된다.

---

## 3. 예측 임계값에 따른 정비 의사결정 변화

RandomForest가 계산한 고장 확률을 기준으로 임계값을 변경한 결과는 다음과 같다.

|     임계값 |    고장 재현율 | FP(오탐) | FN(미탐) |
| ------: | --------: | -----: | -----: |
|     0.2 |     0.929 |     36 |      3 |
| **0.3** | **0.810** | **25** |  **8** |
|     0.4 |     0.738 |     18 |     11 |
| **0.5** | **0.690** | **12** | **13** |
|     0.7 |     0.452 |      1 |     23 |

노트북의 동일한 임계값 계산 기준에서 0.5의 재현율은 약 0.69, FP는 12건, FN은 13건이었고, 0.3에서는 재현율이 약 0.81로 증가하면서 FP 25건, FN 8건으로 나타났다.

### 임계값 0.5 → 0.3 변경 효과

임계값을 0.5에서 0.3으로 낮추면 고장 가능성이 상대적으로 낮게 예측된 설비도 사전에 점검 대상으로 포함된다.

**얻는 효과**

고장 재현율은 약 **69% → 81%**로 약 12%p 상승한다.

미탐(FN)은 **13건 → 8건**으로 감소한다.

즉, 테스트 데이터 기준으로 기존에 놓치던 고장 임박 사례 중 **5건을 추가로 사전에 발견**할 수 있다.

이는 실제 운영 환경에서는 돌발 고장, 생산 중단, 긴급 수리와 같은 다운타임 리스크를 줄이는 방향으로 작용한다.

**발생하는 비용**

반대로 오탐(FP)은 **12건 → 25건**으로 증가한다.

즉, 실제로는 정상인 설비를 점검 대상으로 판단하는 사례가 **13건 증가**한다.

따라서 작업자의 점검 시간, 인력 투입, 설비 정지 및 불필요한 부품 교체 등의 예방정비 비용이 증가할 수 있다.

### 비용 관점의 판단

임계값을 0.5에서 0.3으로 변경하면 테스트 표본 기준으로:

* 추가 오탐: **+13건**
* 감소하는 미탐: **-5건**

이다.

따라서 고장 미탐 1건으로 발생하는 다운타임 비용이 불필요한 예방점검 1건의 비용보다 충분히 크다면 0.3이 경제적으로 유리하다.

단순 비용 비교로 보면,

**미탐 1건의 비용 > 오탐 점검 1건 비용 × 약 2.6**

인 환경에서는 테스트 결과상 0.3으로 임계값을 낮추는 것이 비용 측면에서도 합리적인 선택이 될 가능성이 높다.

다만 실제 최종 임계값은 설비별 고장 손실 비용, 점검 인건비, 부품비, 계획정지 가능 시간 등을 금액으로 환산하여 결정할 필요가 있다.

---

## 4. MIMII 이상탐지 모델 비교

### 4.1 IsolationForest

IsolationForest는 라벨을 학습에 직접 사용하지 않는 비지도 이상탐지 방식으로 적용되었다.

평가 결과 이상 클래스는 다음 성능을 기록하였다.

| 항목           |   결과 |
| ------------ | ---: |
| 이상 Precision | 0.55 |
| 이상 Recall    | 0.25 |
| 이상 F1-score  | 0.35 |
| 전체 Accuracy  | 0.79 |

혼동행렬은 다음과 같다.

* 정상 → 정상: 591건
* 정상 → 이상: 36건
* 이상 → 정상: **129건**
* 이상 → 이상: 44건

즉, 실제 이상 173건 가운데 44건만 탐지하여 이상 재현율이 약 **25%**에 그쳤다.

정비 목적에서는 이상 발생을 조기에 발견하는 것이 핵심인데, 이 모델은 상당수의 이상을 정상으로 판단하므로 단독 운영 모델로 사용하기에는 위험도가 높은 것으로 판단된다.

---

## 5. RandomForest 이상탐지 결과

지도학습 RandomForest는 MIMII 데이터를 학습/테스트 데이터로 분리한 후 평가되었다. 데이터는 테스트 비율 30%, `random_state=42` 및 클래스 비율을 유지하는 stratified split 방식으로 나누었다.

실제 테스트 결과는 다음과 같다.

| 항목           |       결과 |
| ------------ | -------: |
| 이상 Precision | **0.98** |
| 이상 Recall    | **0.88** |
| 이상 F1-score  | **0.93** |
| 전체 Accuracy  | **0.97** |
| FN           |   **6건** |

RandomForest는 테스트 데이터 240건에서 전체 정확도 97%를 기록했으며, 이상 클래스의 정밀도 0.98, 재현율 약 0.88, F1-score 0.93을 기록하였다.

별도의 평가 함수에서도 RandomForest의 재현율은 **0.885**, FN은 **6건**으로 확인되었다.

이는 IsolationForest와 비교했을 때 이상을 실제로 찾아내는 능력이 크게 향상되었음을 의미한다.

---

## 6. IsolationForest vs RandomForest 현장 적용 판단

본 데이터의 결과만을 기준으로 하면 실제 정비 현장에는 **RandomForest 기반 지도학습 모델을 우선 적용하는 것을 권장한다.**

가장 중요한 이유는 이상 재현율의 차이다.

IsolationForest의 이상 재현율은 약 **25%**인 반면 RandomForest는 약 **88%**이다. IsolationForest는 실제 이상 173건 중 129건을 정상으로 판정했지만, RandomForest 테스트에서는 FN이 6건으로 나타났다.
예지보전에서는 정상 설비를 한 번 더 점검하는 비용보다 실제 이상을 놓쳐 설비가 정지하는 위험이 일반적으로 더 중요한 의사결정 요소가 된다. 따라서 현재 분석 데이터에서는 RandomForest가 정비팀의 사전 대응 목적에 더 적합한 모델이다.

다만 두 모델의 평가 방식에는 차이가 있다. IsolationForest 결과는 전체 MIMII 데이터에서 비지도 방식으로 이상을 찾은 뒤 실제 라벨과 비교한 것이고, RandomForest는 라벨이 있는 데이터를 학습/테스트로 분리하여 평가하였다. 따라서 두 숫자를 완전히 동일한 조건의 모델 경쟁 결과로 해석하기보다는 **라벨 데이터가 확보된 경우 지도학습이 매우 높은 성능을 보였다는 결과**로 해석하는 것이 적절하다.

실제 현장에서는 RandomForest를 주 모델로 운용하고, 기존 학습 데이터에 존재하지 않는 새로운 유형의 이상을 탐색하는 보조 수단으로 비지도 이상탐지 모델을 검토할 수 있다.

---

## 7. 권장 임계값

### 최종 권장값: **0.3**

**권장 근거: 임계값을 0.5에서 0.3으로 낮추면 고장 재현율이 약 69%에서 81%로 상승하고 FN이 13건에서 8건으로 감소하여, 추가 점검 비용을 감수하는 대신 돌발 고장 및 다운타임 위험을 줄일 수 있기 때문이다.**

0.2에서는 재현율이 약 93%까지 상승하고 FN이 3건으로 감소하지만 FP가 36건까지 증가한다. 반면 0.4는 FP가 18건으로 감소하지만 FN이 11건으로 증가한다. 따라서 현재 제공된 데이터에서는 **0.3이 고장 미탐 감소와 불필요한 점검 증가 사이의 현실적인 절충점**으로 판단된다.

---

## 8. 정비팀 최종 권고안

**1차 운영 기준**

CMAPSS 고장 예측 알람의 기준 임계값을 **0.3으로 설정**한다.

**정비 우선순위**

고장확률이 0.3 이상인 설비는 예방점검 대상으로 분류하되, 확률값에 따라 우선순위를 추가로 구분하는 운영 방식을 권장한다.

예를 들어 0.5 이상의 설비는 우선 점검 대상으로, 0.3~0.5 구간은 관찰 또는 계획점검 대상으로 운영하면 증가하는 오탐에 따른 정비 부담을 일부 줄일 수 있다.

**이상탐지 모델**

라벨 데이터가 충분한 현재 조건에서는 **RandomForest를 주 이상탐지 모델로 사용하는 것을 권장**한다. IsolationForest는 새로운 패턴을 탐색하는 보조 모델로 활용하는 방안을 검토한다.

**운영 후 재평가**

실제 현장 적용 후에는 FP에 따른 평균 점검비용과 FN에 따른 평균 다운타임 비용을 기록하여 임계값을 다시 최적화해야 한다. 현재 0.3 권고는 제공된 테스트 데이터의 FP/FN 결과를 기준으로 한 초기 운영안이며 실제 설비 비용 데이터가 추가되면 최종 기준을 조정할 필요가 있다.

---

## 9. 결론

현재 CMAPSS 모델은 전체 정확도는 양호하지만 예지보전의 핵심인 고장 임박 탐지 재현율 측면에서는 개선 여지가 있다. 임계값을 0.5에서 0.3으로 낮출 경우 불필요한 점검은 증가하지만 고장 미탐을 줄여 예기치 않은 설비 정지 위험을 낮출 수 있다.

MIMII 데이터에서는 지도학습 RandomForest가 비지도 IsolationForest보다 높은 이상탐지 성능을 보였다. 따라서 **RandomForest 기반 모델 + 0.3 수준의 민감한 알람 기준**을 초기 운영안으로 적용하고, 실제 정비 비용 및 다운타임 데이터를 축적한 후 임계값을 비용 기반으로 재조정하는 전략을 권장한다.
